In [ ]:
import pandas as pd
import numpy as np

ark_final=pd.read_csv('arkansas_integrated.csv')
cal_final=pd.read_csv('california_integrated.csv')
print(ark_final['target_label'])
print(cal_final.columns)

Index(['B2_0', 'B3_0', 'B4_0', 'B5_0', 'B6_0', 'B7_0', 'B8_0', 'B8A_0',
       'B11_0', 'B12_0',
       ...
       'dew_32', 'temp_33', 'precip_33', 'dew_33', 'temp_34', 'precip_34',
       'dew_34', 'temp_35', 'precip_35', 'dew_35'],
      dtype='object', length=513)
Index(['B2_0', 'B3_0', 'B4_0', 'B5_0', 'B6_0', 'B7_0', 'B8_0', 'B8A_0',
       'B11_0', 'B12_0',
       ...
       'dew_32', 'temp_33', 'precip_33', 'dew_33', 'temp_34', 'precip_34',
       'dew_34', 'temp_35', 'precip_35', 'dew_35'],
      dtype='object', length=513)


In [8]:
# Define Feature Groups
# Assumes Sentinel-2 bands are B2_0...B12_35
S2_COLS = [col for col in ark_final.columns if col.startswith(('B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B11', 'B12'))]
CLIMATE_COLS = [col for col in ark_final.columns if col.startswith(('temp', 'precip', 'dew'))]
SOIL_COLS = ['carbon', 'ph','texture'] # Replace with your exact soil column names
TOPO_COLS = ['elevation', 'landforms' ] # Replace with your exact topo column names

def get_xy_split(df, feature_list, target_col='target_label', mask_col='mask'):
    """
    Separates the dataframe into Target, Mask, and Features.
    """
    y = df[target_col].values if target_col in df.columns else None
    mask = df[mask_col].values if mask_col in df.columns else np.ones((len(df), 36)) # Default mask if not present
    X = df[feature_list].values
    
    return X, y, mask

In [21]:
import numpy as np

def get_unified_reshaped(df, temporal_cols, static_cols=None, target_col='label'):
    # 1. Target (y) - Search for common names
    target_names = ['label', 'crop_class', 'class', 'target']
    y_col = next((c for c in target_names if c in df.columns), None)
    y = df[y_col].values if y_col else np.zeros(len(df))
    
    # 2. Mask - Reshape to (N, 36, 1)
    mask_cols = [f'mask_{i}' for i in range(36)]
    mask = df[mask_cols].values.reshape(-1, 36, 1).astype('float32')

    # 3. Temporal Features - Reshape to (N, 36, num_vars)
    num_temp_vars = len(temporal_cols) // 36
    X_temp = df[temporal_cols].values.reshape(-1, 36, num_temp_vars).astype('float32')
    
    # 4. Static Features - Broadcast to match temporal shape
    if static_cols:
        # Get static values (N, num_static)
        X_stat_vals = df[static_cols].values.astype('float32')
        # Repeat the static data for each of the 36 time steps
        X_stat_repeated = np.repeat(X_stat_vals[:, np.newaxis, :], 36, axis=1)
        # Combine into one 3D tensor (N, 36, total_vars)
        X = np.concatenate([X_temp, X_stat_repeated], axis=2)
    else:
        X = X_temp
    
    return X, y, mask

# --- Re-assigning your functions to use the unified logic ---

def load_s2_climate(df):
    return get_unified_reshaped(df, S2_COLS + CLIMATE_COLS)

def load_s2_soil(df):
    return get_unified_reshaped(df, S2_COLS, static_cols=SOIL_COLS)

def load_s2_topography(df):
    return get_unified_reshaped(df, S2_COLS, static_cols=TOPO_COLS)

def load_s2_all(df):
    # This will combine S2 + Climate into the temporal block, 
    # then append Soil + Topo as constant features across time
    return get_unified_reshaped(df, S2_COLS + CLIMATE_COLS, static_cols=SOIL_COLS + TOPO_COLS)

In [10]:
def generate_quality_mask(X_s2):
    """
    Generates a boolean mask where a time-step is 'False' if the 
    Sentinel values are 0 or NaN (common for cloud-cleared pixels).
    """
    # Reshape to (Samples, TimeSteps, Bands) -> (10000, 36, 10)
    X_reshaped = X_s2.reshape(X_s2.shape[0], 36, -1)
    
    # Mask is True if any band in that time step has a valid value (> 0)
    mask = np.any(X_reshaped > 0, axis=2) 
    return mask.astype(np.float32)

# raw data + soil 

In [22]:
X, y, mask = load_s2_soil(ark_final)

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"Mask shape: {mask.shape}")

Features shape: (10000, 36, 13)
Target shape: (10000,)
Mask shape: (10000, 36, 1)


# raw data + topography 

In [23]:
X, y, mask = load_s2_topography(ark_final)

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"Mask shape: {mask.shape}")

Features shape: (10000, 36, 12)
Target shape: (10000,)
Mask shape: (10000, 36, 1)


# raw data + climate

In [24]:
X, y, mask = load_s2_climate(ark_final)

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"Mask shape: {mask.shape}")

Features shape: (10000, 36, 13)
Target shape: (10000,)
Mask shape: (10000, 36, 1)


# raw data + soil + topography + climate

In [25]:
X, y, mask = load_s2_all(ark_final)

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"Mask shape: {mask.shape}")

Features shape: (10000, 36, 18)
Target shape: (10000,)
Mask shape: (10000, 36, 1)
